# Benchmark-Relative Tear Sheet

**Use after source analytics:** Reporting notebooks render results produced by the pricing, analytics, statement, and portfolio notebooks; they do not replace those calculation workflows.

**Purpose:** Package alpha, beta, capture, rolling greeks, and multi-factor relative performance into a benchmark review page.

**Prerequisites:** `03_analytics/performance_analytics.ipynb` and `03_analytics/risk_and_factor_analytics.ipynb`.

**What you'll learn:**

- Prepare benchmark-relative performance inputs.
- Render `reporting.benchmark_tearsheet`.
- Use tear sheets as presentation wrappers over analytics outputs.

Build a fund-vs-benchmark `Performance`, then render a `benchmark_tearsheet` —
alpha/beta regression, active risk/return, capture ratios, rolling alpha & beta,
relative-return and relative-drawdown series, and a multi-factor attribution.


In [1]:
import datetime as dt
import random

import pandas as pd

from finstack_quant import reporting
from finstack_quant.analytics import Performance

random.seed(7)
dates = pd.bdate_range("2024-01-02", periods=252)
bench = [random.gauss(0.0004, 0.01) for _ in range(252)]
fund = [0.00030 + 0.92 * bench[i] + random.gauss(0.0, 0.0035) for i in range(252)]
df = pd.DataFrame({"Global Equity Fund": fund, "MSCI World": bench}, index=dates)
perf = Performance.from_returns(df, benchmark_ticker="MSCI World", freq="daily")
print("Tickers:", perf.ticker_names, "benchmark idx:", perf.benchmark_idx)

Tickers: ['Global Equity Fund', 'MSCI World'] benchmark idx: 1


## Benchmark-relative tear sheet

In [2]:
reporting.benchmark_tearsheet(perf, generated=dt.date(2026, 6, 23))

Alpha (ann.),+5.2%
Beta,0.92
Beta 95% CI,"[0.88, 0.96]"
R-Squared,0.87
Adjusted R-Squared,0.87
Tracking Error,5.8%
Information Ratio,0.61
Treynor,+26.2%
M-Squared,+24.5%
Up Capture,0.93
Down Capture,0.98


## With a multi-factor attribution

Pass a pre-computed `multi_factor_greeks` result and factor labels.

In [3]:
smb = [random.gauss(0.0, 0.006) for _ in range(252)]
hml = [random.gauss(0.0, 0.006) for _ in range(252)]
mf = perf.multi_factor_greeks(0, [bench, smb, hml])
reporting.benchmark_tearsheet(
    perf, multi_factor=mf, factor_names=["Market", "Size", "Value"],
    sections=["summary", "multifactor"], generated=dt.date(2026, 6, 23),
)

Alpha (ann.),+5.2%
Beta,0.92
Beta 95% CI,"[0.88, 0.96]"
R-Squared,0.87
Adjusted R-Squared,0.87
Tracking Error,5.8%
Information Ratio,0.61
Treynor,+26.2%
M-Squared,+24.5%
Up Capture,0.93
Down Capture,0.98


## Saving a standalone HTML file

```python
ts = reporting.benchmark_tearsheet(perf, generated=dt.date(2026, 6, 23))
ts.save("benchmark_tearsheet.html")
```

## Takeaways

- Reporting functions are presentation wrappers over analytics, valuation, statement, or portfolio results produced earlier in the curriculum.
- Keep the analytical source of truth in the typed objects or JSON specs, then render a tear sheet for review.
- Pass fixed `generated` dates in examples so notebook output remains reproducible.
